# 🌿 GreenWise — Entraînement du modèle de tri des déchets

Ce notebook entraîne gratuitement une IA qui reconnaît **12 types de déchets** à partir d'une photo, avec **~15 500 images** (dataset Kaggle *Garbage Classification*), puis l'exporte en **TensorFlow Lite** pour l'application mobile.

**Avant de commencer**
1. `Exécution` → `Modifier le type d'exécution` → **GPU (T4)** (gratuit).
2. Créez un compte gratuit sur kaggle.com → *Settings* → *Create New Token* : cela télécharge `kaggle.json` (vous l'enverrez à l'étape 2).
3. Exécutez les cellules dans l'ordre (≈ 20–30 min au total).

Classes : battery, biological, brown-glass, cardboard, clothes, green-glass, metal, paper, plastic, shoes, trash, white-glass.

## 1. Préparation

In [ ]:
import os, shutil, random, numpy as np, matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow :", tf.__version__)
print("GPU :", tf.config.list_physical_devices('GPU') or "AUCUN (activez le GPU !)")

SEED = 42
IMG_SIZE = (224, 224)
BATCH = 32
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

## 2. Télécharger les données (Kaggle)

In [ ]:
from google.colab import files

print("➡️ Choisissez votre fichier kaggle.json")
files.upload()
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

!kaggle datasets download -d mostafaabla/garbage-classification -p /content/data --unzip -q
print("✅ Données téléchargées")

In [ ]:
IMG_EXT = ('.jpg', '.jpeg', '.png', '.bmp')

def find_root(base):
    """Trouve le dossier qui contient un sous-dossier par classe."""
    best, best_n = None, 0
    for d, subs, _ in os.walk(base):
        ok = [s for s in subs if any(f.lower().endswith(IMG_EXT) for f in os.listdir(os.path.join(d, s)))]
        if len(ok) >= 5 and len(ok) == len(subs) and len(ok) > best_n:
            best, best_n = d, len(ok)
    return best

DATA_DIR = find_root('/content/data')
print("Dossier des classes :", DATA_DIR)

### (Optionnel) Ajouter vos propres photos — ex. **électronique** 📱
Le dataset n'a pas de classe « électronique ». Pour en ajouter (ou ajouter des déchets de votre pays) :
1. Dans Google Drive, créez `MyDrive/dechets_extra/electronic/` (et d'autres dossiers de classes si vous voulez) puis mettez-y vos photos (idéalement 300+ par classe).
2. Mettez `USE_EXTRA = True` et exécutez la cellule.

In [ ]:
USE_EXTRA = False   # ← mettez True pour ajouter vos propres images

if USE_EXTRA:
    from google.colab import drive
    drive.mount('/content/drive')
    extra = '/content/drive/MyDrive/dechets_extra'
    for cls in os.listdir(extra):
        src = os.path.join(extra, cls)
        if not os.path.isdir(src): continue
        dst = os.path.join(DATA_DIR, cls)
        os.makedirs(dst, exist_ok=True)
        n = 0
        for f in os.listdir(src):
            if f.lower().endswith(IMG_EXT):
                shutil.copy(os.path.join(src, f), os.path.join(dst, f'extra_{f}')); n += 1
        print(f"  + {n} images ajoutées à la classe '{cls}'")

## 3. Explorer les données

In [ ]:
counts = {c: len([f for f in os.listdir(os.path.join(DATA_DIR, c)) if f.lower().endswith(IMG_EXT)])
          for c in sorted(os.listdir(DATA_DIR)) if os.path.isdir(os.path.join(DATA_DIR, c))}
total = sum(counts.values())
print(f"Total : {total} images, {len(counts)} classes")
plt.figure(figsize=(10, 4))
plt.bar(counts.keys(), counts.values(), color='#43A047')
plt.xticks(rotation=45, ha='right'); plt.title("Images par classe"); plt.tight_layout(); plt.show()

## 4. Créer les jeux d'entraînement / validation

In [ ]:
train_ds = keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset='training', seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH, label_mode='categorical')
val_ds = keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset='validation', seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH, label_mode='categorical')

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)
print("Classes :", class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

# Poids de classes : compense les classes rares (ex. 'clothes' est très fréquente)
class_weight = {i: total / (NUM_CLASSES * counts[c]) for i, c in enumerate(class_names)}
print({class_names[i]: round(w, 2) for i, w in class_weight.items()})

plt.figure(figsize=(10, 6))
for imgs, labels in train_ds.take(1):
    for i in range(9):
        plt.subplot(3, 3, i + 1)
        plt.imshow(imgs[i].numpy().astype('uint8'))
        plt.title(class_names[int(tf.argmax(labels[i]))]); plt.axis('off')
plt.show()

## 5. Construire le modèle (MobileNetV2 pré-entraîné sur ImageNet)

In [ ]:
augment = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.2),
], name='augmentation')

base = keras.applications.MobileNetV2(input_shape=IMG_SIZE + (3,), include_top=False, weights='imagenet')
base.trainable = False

inputs = keras.Input(shape=IMG_SIZE + (3,), name='image')       # pixels 0..255
x = augment(inputs)
x = layers.Rescaling(1./127.5, offset=-1)(x)                    # normalisation intégrée au modèle
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax', name='waste_class')(x)
model = keras.Model(inputs, outputs)

model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

## 6. Entraînement — phase 1 (couches de base gelées)

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True, monitor='val_accuracy'),
    keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=2, monitor='val_loss'),
]
h1 = model.fit(train_ds, validation_data=val_ds, epochs=8,
               class_weight=class_weight, callbacks=callbacks)

## 7. Entraînement — phase 2 (fine-tuning pour gagner en précision)

In [ ]:
base.trainable = True
for layer in base.layers[:-40]:      # on ne réentraîne que les 40 dernières couches
    layer.trainable = False

model.compile(optimizer=keras.optimizers.Adam(1e-5),
              loss='categorical_crossentropy', metrics=['accuracy'])
h2 = model.fit(train_ds, validation_data=val_ds, epochs=12,
               class_weight=class_weight, callbacks=callbacks)

acc = h1.history['accuracy'] + h2.history['accuracy']
vacc = h1.history['val_accuracy'] + h2.history['val_accuracy']
plt.plot(acc, label='entraînement'); plt.plot(vacc, label='validation')
plt.axvline(len(h1.history['accuracy']) - 1, color='gray', ls='--')
plt.legend(); plt.title('Précision'); plt.xlabel('époque'); plt.show()

## 8. Évaluer le modèle

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

y_true, y_pred = [], []
for imgs, labels in val_ds:
    p = model.predict(imgs, verbose=0)
    y_pred += list(np.argmax(p, axis=1))
    y_true += list(np.argmax(labels.numpy(), axis=1))

print(classification_report(y_true, y_pred, target_names=class_names, digits=3))
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Prédit'); plt.ylabel('Réel'); plt.show()

## 9. Exporter pour l'application mobile (TensorFlow Lite)

In [ ]:
model.save('waste_model.keras')

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]        # modèle ~4x plus léger
tflite_model = converter.convert()
with open('waste_model.tflite', 'wb') as f:
    f.write(tflite_model)
with open('labels.txt', 'w') as f:
    f.write('\n'.join(class_names))
print(f"✅ waste_model.tflite : {len(tflite_model)/1e6:.1f} Mo")

# Test rapide du modèle .tflite
interp = tf.lite.Interpreter(model_content=tflite_model)
interp.allocate_tensors()
inp, out = interp.get_input_details()[0], interp.get_output_details()[0]
imgs, labels = next(iter(val_ds))
ok = 0
n = min(16, len(imgs))
for i in range(n):
    interp.set_tensor(inp['index'], imgs[i:i+1].numpy().astype('float32'))
    interp.invoke()
    ok += int(np.argmax(interp.get_tensor(out['index'])) == int(tf.argmax(labels[i])))
print(f"Test TFLite : {ok}/{n} corrects")

In [ ]:
from google.colab import files
files.download('waste_model.tflite')
files.download('labels.txt')
# (facultatif) copie de sauvegarde sur Drive :
# from google.colab import drive; drive.mount('/content/drive')
# !cp waste_model.tflite labels.txt /content/drive/MyDrive/

## 10. Brancher le modèle dans GreenWise

1. Copiez `waste_model.tflite` et `labels.txt` dans le dossier `assets/` de l'application Flutter.
2. Les 12 classes du modèle correspondent aux 8 types de l'application ainsi :

| Classe du modèle | Type GreenWise |
|---|---|
| plastic | Plastique |
| brown-glass, green-glass, white-glass | Verre |
| paper, cardboard | Papier / Carton |
| metal | Métal |
| biological | Organique |
| battery | Dangereux |
| clothes, shoes, trash | Autre |
| *(vos images « electronic » si ajoutées)* | Électronique |

3. Dites-moi quand vous avez le fichier `.tflite` : je modifie `classifier_service.dart` pour l'utiliser dans l'application à la place de ML Kit.

> 💡 Plus il y a de photos de **vos** déchets locaux (emballages du pays, marchés…), meilleure sera la précision réelle.